# Re-run: demographic groups on ASAP 2.0 (Paper VI, RQ6)

This notebook recomputes the RQ6 table's measured rows: the label effect (high − low) on all essays, the
low-label cost difference for each focal group, the race/ethnicity widest gap against its bar, and the
observed no-label residual gap by group, for Qwen3-235B and GPT-4.1-mini. It also checks the two
contrasts the text reports beside them (the high − low spread by group, and low against the mid label).

**Inputs.** The per-case model outputs published with release `ai-grader-reliability-2026`
(`per-case-model-outputs.jsonl` or the `.gz` the report offers, in this folder or in a folder named by
`RELEASE_DIR`), **and the public ASAP 2.0 corpus**. The published file deliberately carries no
demographic field. The group fields are public in the corpus and re-join by `essay_id`, so this notebook
needs `ASAP2_train_sourcetexts.csv` from ASAP 2.0 (Crossley et al. 2025, Kaggle `lburleigh/asap-2-0`,
version 8, CC BY 4.0; its hash is in the release file). Set `ASAP2_DIR` to the folder that holds it, or
put the file beside this notebook. The notebook stops with a message saying so if it is absent. It reads
only the id, score and the five group columns, never the essay text, and writes nothing. Python 3.9+
with numpy.

The corrected rows of the table (the gap at equal true quality, at reliability 0.75 and 0.86) come from
a separate measurement-error model (`studies/measurement/grader-matching-artefact/`, in the release's
code package) and are not recomputed here.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "ai-grader-reliability-2026"   # filled in by board/render_report.py
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Data:", shown(find_release_file()))

Data: per-case-model-outputs.jsonl


In [2]:
import csv
import numpy as np

RUNS = {"GPT-4.1-mini": "grader-vi-groups/run-gpt41mini", "Qwen3-235B": "grader-vi-groups/run-qwen"}
AXES = {"ell": ("Yes", "No"),
        "economic": ("Economically disadvantaged", "Not economically disadvantaged"),
        "disability": ("Identified as having disability", "Not identified as having disability"),
        "gender": ("F", "M")}
COLUMNS = {"ell": "ell_status", "economic": "economically_disadvantaged",
           "disability": "student_disability_status", "race": "race_ethnicity", "gender": "gender"}

path = corpus_file("ASAP2_DIR", "ASAP2_train_sourcetexts.csv")
csv.field_size_limit(1 << 30)
corpus = {}
with open(path, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        corpus[row["essay_id"]] = {k: (row[c].strip() or "NA") for k, c in COLUMNS.items()}
print(f"{len(corpus)} essays in the corpus")

24728 essays in the corpus


**Design weights.** The draw over-sampled English learners (ELL) and disability-identified students in
three quota cells (ELL; not ELL but disability-identified; everyone else). Every mean is weighted by
the cell's share of the corpus over its share of the draw, which restores the corpus's own mix
(`grader-attainment-groups/analyse.py`, `design_weights`).

In [3]:
def cell(g):
    return "ell" if g["ell"] == "Yes" else "disability" if g["disability"] == AXES["disability"][0] else "rest"

corpus_share = {c: np.mean([cell(g) == c for g in corpus.values()]) for c in ("ell", "disability", "rest")}

class Run:
    def __init__(self, run):
        rows = list(per_case({run}))
        missing = [r["id"] for r in rows if str(r["id"]) not in corpus]
        assert not missing, f"{len(missing)} essay ids not in the corpus file: is it ASAP 2.0 version 8?"
        g = [corpus[str(r["id"])] for r in rows]
        self.groups = {k: np.array([x[k] for x in g]) for k in COLUMNS}
        self.h = np.array([r["human"] for r in rows], float)
        self.stratum = np.clip(self.h.astype(int), 1, 5) - 1          # human score strata 1, 2, 3, 4, 5-6
        self.arms = {a: np.array([np.nan if r.get(a) is None else r[a] for r in rows], float)
                     for a in ("none", "low", "high", "mid") if any(r.get(a) is not None for r in rows)}
        cells = np.array([cell(x) for x in g])
        self.wt = np.ones(len(rows))
        for c, share in corpus_share.items():
            self.wt[cells == c] = share / np.mean(cells == c)
        self.complete = np.all([np.isfinite(self.arms[a]) for a in ("none", "low", "high")], axis=0)

data = {m: Run(run) for m, run in RUNS.items()}
for m, d in data.items():
    print(f"{m}: {len(d.h)} essays, {d.complete.sum()} graded in every arm; arms {sorted(d.arms)}")

GPT-4.1-mini: 3000 essays, 2935 graded in every arm; arms ['high', 'low', 'none']
Qwen3-235B: 3000 essays, 2997 graded in every arm; arms ['high', 'low', 'mid', 'none']


**The estimator.** For a per-essay contrast y (for example low − none), the group difference is focal
minus reference **at equal human score**: the weighted mean of y in each human-score stratum, the
difference between the groups in each stratum, averaged with the strata's pooled weight. A stratum is
used only if both groups have at least 5 essays in it. Intervals are ±1.96 bootstrap SDs, resampling
essays within group × stratum, 2,000 times; the label effect's interval resamples all essays. All of
this is `grader-attainment-groups/analyse.py` (`strat_diff`, `boot_sd`, `gate`, `race_axis`), written
out, with the study's seed and order of draws.

In [4]:
def cell_stats(y, strat, mask, wt):
    n = np.bincount(strat[mask], minlength=5).astype(float)
    W = np.bincount(strat[mask], weights=wt[mask], minlength=5)
    with np.errstate(invalid="ignore", divide="ignore"):
        mu = np.bincount(strat[mask], weights=wt[mask] * y[mask], minlength=5) / W
        q = np.bincount(strat[mask], weights=(wt[mask] * (y[mask] - mu[strat[mask]])) ** 2, minlength=5)
        vm = np.where(n > 1, q / W ** 2 * n / (n - 1), np.nan)
    return n, W, mu, vm

def strat_diff(y, strat, isg, isr, wt):
    ng, Wg, mg, _ = cell_stats(y, strat, isg, wt)
    nr, Wr, mr, _ = cell_stats(y, strat, isr, wt)
    keep = (ng >= 5) & (nr >= 5)
    om = (Wg + Wr) * keep
    om = om / om.sum()
    return float(np.sum(om[keep] * (mg[keep] - mr[keep]))), keep

def boot_mean(y, wt, idx, rng, reps=2000):
    pick = idx[rng.integers(0, len(idx), (reps, len(idx)))]
    return (y[pick] * wt[pick]).sum(1) / wt[pick].sum(1)

def boot_sd(y, strat, isg, isr, keep, wt, rng):
    ig = [np.flatnonzero(isg & (strat == s)) for s in range(5)]
    ir = [np.flatnonzero(isr & (strat == s)) for s in range(5)]
    om = (np.array([wt[i].sum() for i in ig]) + np.array([wt[i].sum() for i in ir])) * keep
    om = om / om.sum()
    out = np.zeros(2000)
    for s in np.flatnonzero(keep):
        out += om[s] * (boot_mean(y, wt, ig[s], rng) - boot_mean(y, wt, ir[s], rng))
    return float(out.std(ddof=1))

EMAX = [0, 0, 0.56, 0.85, 1.03, 1.16, 1.27, 1.35, 1.42, 1.49, 1.54]   # expected max of k normals

def race_axis(d, y, rng):
    """Widest gap between race/ethnicity levels (>= 40 essays), each standardised to the pooled score mix."""
    g, ok = d.groups["race"], d.complete & np.isfinite(y)
    levels = sorted({x for x in g[ok] if x != "NA"}, key=lambda x: -int(np.sum(ok & (g == x))))
    levels = [x for x in levels if np.sum(ok & (g == x)) >= 40]
    while True:
        masks = {x: ok & (g == x) for x in levels}
        counts = np.array([np.bincount(d.stratum[m], minlength=5) for m in masks.values()])
        keep = (counts >= 5).all(0)
        if counts[:, keep].sum() / counts.sum() >= 0.8:
            break
        levels.pop()
    allm = np.any(list(masks.values()), axis=0)
    w = np.bincount(d.stratum[allm], weights=d.wt[allm], minlength=5) * keep
    w = w / w.sum()
    std = {}
    for x, m in masks.items():
        _, _, mu, _ = cell_stats(y, d.stratum, m, d.wt)
        idx = [np.flatnonzero(m & (d.stratum == s)) for s in range(5)]
        acc = np.zeros(2000)
        for s in np.flatnonzero(keep):
            acc += w[s] * boot_mean(y, d.wt, idx[s], rng)
        std[x] = (float(np.sum(w[keep] * mu[keep])), float(acc.std(ddof=1)))
    lo, hi = min(std, key=lambda x: std[x][0]), max(std, key=lambda x: std[x][0])
    sd = float(np.hypot(std[hi][1], std[lo][1]))
    pairs = len(levels) * (len(levels) - 1) // 2
    return {"gap": std[hi][0] - std[lo][0], "bar": (2 + EMAX[min(pairs, 10)]) * sd}

CONTRASTS = {"D_low": ("low", "none"), "D_spread": ("high", "low"), "D_high": ("high", "none"), "D_lowmid": ("low", "mid")}

def analyse(d, seed=20261005):
    rng = np.random.default_rng(seed)
    c = d.complete
    y = (d.arms["high"] - d.arms["low"])[c]
    S = float(np.sum(d.wt[c] * y) / d.wt[c].sum())
    sd = float(boot_mean(y, d.wt[c], np.arange(len(y)), rng).std(ddof=1))
    out = {"S": (S, S - 1.96 * sd, S + 1.96 * sd)}
    for name, (a, b) in CONTRASTS.items():
        if a not in d.arms or b not in d.arms:
            continue
        y = d.arms[a] - d.arms[b]
        ok = d.complete & np.isfinite(y)
        for ax, (f, r) in AXES.items():
            isg, isr = ok & (d.groups[ax] == f), ok & (d.groups[ax] == r)
            diff, keep = strat_diff(y, d.stratum, isg, isr, d.wt)
            s = boot_sd(y, d.stratum, isg, isr, keep, d.wt, rng)
            out[name, ax] = (diff, diff - 1.96 * s, diff + 1.96 * s, diff / s)
        out[name, "race"] = race_axis(d, y, rng)
    resid = d.arms["none"] - d.h
    for ax, (f, r) in AXES.items():
        out["none_gap", ax] = strat_diff(resid, d.stratum, c & (d.groups[ax] == f), c & (d.groups[ax] == r), d.wt)[0]
    return out

res = {m: analyse(d) for m, d in data.items()}

In [5]:
def fmt(t):
    return f"{t[0]:+.3f} [{t[1]:+.3f}, {t[2]:+.3f}]"

LABELS = {"ell": "English learners", "economic": "economically disadvantaged", "disability": "disability-identified",
          "gender": "female"}
print(f"{'':52s} {'GPT-4.1-mini':>26s} {'Qwen3-235B':>26s}")
print(f"{'Label effect, high − low (all essays)':52s} " + " ".join(f"{fmt(res[m]['S']):>26s}" for m in RUNS))
for ax in AXES:
    print(f"{'Low-label cost difference, ' + LABELS[ax]:52s} " + " ".join(f"{fmt(res[m]['D_low', ax]):>26s}" for m in RUNS))
print(f"{'Race/ethnicity, widest gap against its bar':52s} " +
      " ".join(f"{res[m]['D_low', 'race']['gap']:.3f} against {res[m]['D_low', 'race']['bar']:.3f}".rjust(26) for m in RUNS))
print(f"{'No-label residual gap, ELL / economic / disability':52s} " +
      " ".join(" / ".join(f"{res[m]['none_gap', ax]:+.2f}" for ax in ("ell", "economic", "disability")).rjust(26) for m in RUNS))

                                                                   GPT-4.1-mini                 Qwen3-235B
Label effect, high − low (all essays)                   +0.225 [+0.206, +0.245]    +0.484 [+0.460, +0.508]
Low-label cost difference, English learners             +0.034 [-0.004, +0.073]    +0.075 [+0.029, +0.120]
Low-label cost difference, economically disadvantaged    +0.062 [+0.017, +0.106]    +0.024 [-0.029, +0.078]
Low-label cost difference, disability-identified        +0.023 [-0.016, +0.063]    +0.105 [+0.056, +0.153]
Low-label cost difference, female                       -0.034 [-0.074, +0.006]    -0.033 [-0.083, +0.017]
Race/ethnicity, widest gap against its bar                  0.047 against 0.181        0.094 against 0.201
No-label residual gap, ELL / economic / disability        -0.21 / -0.13 / -0.29      -0.24 / -0.17 / -0.32


In [6]:
# The paper's printed values (RQ6 table). Point estimates must match exactly, as printed.
def same(x, printed, nd):
    return f"{x:.{nd}f}" == f"{printed:.{nd}f}"

PRINTED = {
    "GPT-4.1-mini": {"S": 0.225, "ell": 0.034, "economic": 0.062, "disability": 0.023, "gender": -0.034,
                     "race": (0.047, 0.181), "none": (-0.21, -0.13, -0.29)},
    "Qwen3-235B": {"S": 0.484, "ell": 0.075, "economic": 0.024, "disability": 0.105, "gender": -0.033,
                   "race": (0.094, 0.201), "none": (-0.24, -0.17, -0.32)},
}
for m, p in PRINTED.items():
    R = res[m]
    assert same(R["S"][0], p["S"], 3), (m, R["S"])
    for ax in AXES:
        assert same(R["D_low", ax][0], p[ax], 3), (m, ax, R["D_low", ax])
    assert same(R["D_low", "race"]["gap"], p["race"][0], 3), (m, R["D_low", "race"])
    for ax, v in zip(("ell", "economic", "disability"), p["none"]):
        assert same(R["none_gap", ax], v, 2), (m, ax, R["none_gap", ax])

# The text: the spread (high − low) by group stays within ±0.045, GPT-4.1-mini's economic spread is
# −0.045, and on Qwen the low label against the same-length mid label takes 0.06 to 0.08 more off the
# three focal groups.
spread = [res[m]["D_spread", ax][0] for m in RUNS for ax in AXES]
assert max(abs(x) for x in spread) <= 0.0455 and same(res["GPT-4.1-mini"]["D_spread", "economic"][0], -0.045, 3)
lowmid = [res["Qwen3-235B"]["D_lowmid", ax][0] for ax in ("ell", "economic", "disability")]
assert same(min(-x for x in lowmid), 0.06, 2) and same(max(-x for x in lowmid), 0.08, 2), lowmid
print(f"spread differences within ±{max(abs(x) for x in spread):.3f}; Qwen low vs mid: "
      + ", ".join(f"{x:+.3f}" for x in lowmid))
print("All 18 printed point values of the measured rows, and the spread and mid-label figures, match the paper.")

spread differences within ±0.045; Qwen low vs mid: -0.079, -0.082, -0.063
All 18 printed point values of the measured rows, and the spread and mid-label figures, match the paper.


**Intervals and verdicts.** The cell above uses the study's seed (20261005) and its order of draws
(the label effect, then every contrast's axes and race in turn), so on the same numpy version the
intervals are the study's own; they are compared with the printed ones here rather than asserted,
because another numpy version can draw a different random stream. A difference is DIFFERENTIAL when
|z| exceeds the Bonferroni bar over five axes, z\* = 2.576; every resolved difference favours the focal
group (z > 0).

In [7]:
PRINTED_CI = {
    ("GPT-4.1-mini", "S"): (0.206, 0.245), ("Qwen3-235B", "S"): (0.460, 0.508),
    ("GPT-4.1-mini", "ell"): (-0.004, 0.073), ("Qwen3-235B", "ell"): (0.029, 0.120),
    ("GPT-4.1-mini", "economic"): (0.017, 0.106), ("Qwen3-235B", "economic"): (-0.029, 0.078),
    ("GPT-4.1-mini", "disability"): (-0.016, 0.063), ("Qwen3-235B", "disability"): (0.056, 0.153),
    ("GPT-4.1-mini", "gender"): (-0.074, 0.006), ("Qwen3-235B", "gender"): (-0.083, 0.017),
}
match = True
for (m, k), (lo, up) in PRINTED_CI.items():
    got = res[m]["S"] if k == "S" else res[m]["D_low", k]
    ok = same(got[1], lo, 3) and same(got[2], up, 3)
    match &= ok
    z = "" if k == "S" else f"   z {got[3]:+.2f}{'  DIFFERENTIAL' if abs(got[3]) > 2.576 else ''}"
    print(f"{m:13s} {k:10s} {fmt(got)}   paper [{lo:+.3f}, {up:+.3f}]{z}")
for m in RUNS:
    r_ = res[m]["D_low", "race"]
    print(f"{m:13s} race: widest gap {r_['gap']:.3f} against bar {r_['bar']:.3f}")
print("Intervals equal the printed ones:", match)
eco = res["GPT-4.1-mini"]["D_spread", "economic"]
print(f"GPT-4.1-mini spread, economic: {fmt(eco)}   paper −0.045 [−0.088, −0.002]")
assert all(res[m]["D_low", ax][3] > -2.576 for m in RUNS for ax in AXES), "a differential against a focal group"

GPT-4.1-mini  S          +0.225 [+0.206, +0.245]   paper [+0.206, +0.245]
Qwen3-235B    S          +0.484 [+0.460, +0.508]   paper [+0.460, +0.508]
GPT-4.1-mini  ell        +0.034 [-0.004, +0.073]   paper [-0.004, +0.073]   z +1.77
Qwen3-235B    ell        +0.075 [+0.029, +0.120]   paper [+0.029, +0.120]   z +3.20  DIFFERENTIAL
GPT-4.1-mini  economic   +0.062 [+0.017, +0.106]   paper [+0.017, +0.106]   z +2.69  DIFFERENTIAL
Qwen3-235B    economic   +0.024 [-0.029, +0.078]   paper [-0.029, +0.078]   z +0.88
GPT-4.1-mini  disability +0.023 [-0.016, +0.063]   paper [-0.016, +0.063]   z +1.16
Qwen3-235B    disability +0.105 [+0.056, +0.153]   paper [+0.056, +0.153]   z +4.20  DIFFERENTIAL
GPT-4.1-mini  gender     -0.034 [-0.074, +0.006]   paper [-0.074, +0.006]   z -1.68
Qwen3-235B    gender     -0.033 [-0.083, +0.017]   paper [-0.083, +0.017]   z -1.28
GPT-4.1-mini  race: widest gap 0.047 against bar 0.181
Qwen3-235B    race: widest gap 0.094 against bar 0.201
Intervals equal the printed 